# Weather Data Pipeline Walkthrough
This notebook demonstrates the end-to-end weather data pipeline using the same ingestion code used by the Airflow DAG.

**Flow:** Open-Meteo API ? PostgreSQL raw layer ? dbt staging ? dbt mart.

## 1. Load weather data
The pipeline is configured through config/cities.yml. The notebook imports the same load_weather function used by Airflow.

In [1]:
import sys
sys.path.insert(0, '/opt/airflow')
from ingestion.weather_pipeline import load_weather

run_date = '2026-09-19'
load_weather(run_date)
print(f'Loaded weather data for {run_date}')

Loaded weather data for 2026-09-19


## 2. Check raw row count

In [2]:
import psycopg2
import pandas as pd

connection = psycopg2.connect(
    host='postgres', port=5432, user='de', password='de', dbname='warehouse'
)

count_df = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM raw_weather_daily WHERE date = '2026-09-19';",
    connection
)
count_df

/tmp/ipykernel_80/2294166260.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count_df = pd.read_sql_query(


,row_count
0,5


## 3. Sample raw weather data

In [3]:
raw_df = pd.read_sql_query(
    "SELECT * FROM raw_weather_daily WHERE date = '2026-09-19' ORDER BY city_name;",
    connection
)
raw_df

/tmp/ipykernel_80/864839691.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw_df = pd.read_sql_query(


,city_name,latitude,longitude,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,Bengaluru,12.9716,77.5946,2026-09-19,30.5,20.7,25.3,11.1,9.4
1,Chennai,13.0827,80.2707,2026-09-19,33.9,26.1,29.6,4.8,15.3
2,Delhi,28.6139,77.2090,2026-09-19,32.8,25.0,28.9,0.0,5.8
3,Hyderabad,17.3850,78.4867,2026-09-19,32.1,25.2,28.6,1.1,8.0
4,Mumbai,19.0760,72.8777,2026-09-19,30.3,24.7,27.5,6.4,13.8


## 4. Run dbt transformations
The dbt project builds the staging model and business-facing daily mart.

In [4]:
import subprocess

result = subprocess.run(
    ['dbt', 'run', '--project-dir', '/opt/airflow/dbt', '--profiles-dir', '/opt/airflow/dbt'],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

13:21:15  Running with dbt=1.8.8
13:21:15  Registered adapter: postgres=1.8.2
13:21:17  Found 2 models, 8 data tests, 1 source, 423 macros
13:21:17  
13:21:17  Concurrency: 2 threads (target='dev')
13:21:17  
13:21:17  1 of 2 START sql view model analytics.stg_weather_daily ........................ [RUN]
13:21:18  1 of 2 OK created sql view model analytics.stg_weather_daily ................... [CREATE VIEW in 0.42s]
13:21:18  2 of 2 START sql table model analytics.fct_city_daily .......................... [RUN]
13:21:18  2 of 2 OK created sql table model analytics.fct_city_daily ..................... [SELECT 10 in 0.43s]
13:21:18  
13:21:18  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 1.35 seconds (1.35s).
13:21:18  
13:21:18  Completed successfully
13:21:18  
13:21:18  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2



## 5. Run dbt data quality tests

In [5]:
result = subprocess.run(
    ['dbt', 'test', '--project-dir', '/opt/airflow/dbt', '--profiles-dir', '/opt/airflow/dbt'],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

13:21:26  Running with dbt=1.8.8
13:21:26  Registered adapter: postgres=1.8.2
13:21:28  Found 2 models, 8 data tests, 1 source, 423 macros
13:21:28  
13:21:28  Concurrency: 2 threads (target='dev')
13:21:28  
13:21:28  1 of 8 START test not_null_fct_city_daily_city_name ............................ [RUN]
13:21:28  2 of 8 START test not_null_fct_city_daily_date ................................. [RUN]
13:21:28  2 of 8 PASS not_null_fct_city_daily_date ....................................... [PASS in 0.39s]
13:21:28  1 of 8 PASS not_null_fct_city_daily_city_name .................................. [PASS in 0.42s]
13:21:28  3 of 8 START test not_null_stg_weather_daily_city_name ......................... [RUN]
13:21:28  4 of 8 START test not_null_stg_weather_daily_date .............................. [RUN]
13:21:29  3 of 8 PASS not_null_stg_weather_daily_city_name ............................... [PASS in 0.17s]
13:21:29  4 of 8 PASS not_null_stg_weather_daily_date ............................

## 6. Prove idempotency
The same logical date is loaded again. The loader deletes existing rows for that date before inserting the refreshed data, so the row count should remain unchanged.

In [6]:
before = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM raw_weather_daily WHERE date = '2026-09-19';",
    connection
).iloc[0, 0]

load_weather('2026-09-19')

after = pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM raw_weather_daily WHERE date = '2026-09-19';",
    connection
).iloc[0, 0]

print(f'Rows before rerun: {before}')
print(f'Rows after rerun:  {after}')
print(f'Idempotent: {before == after}')

/tmp/ipykernel_80/3539860190.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  before = pd.read_sql_query(


Rows before rerun: 5
Rows after rerun:  5
Idempotent: True


/tmp/ipykernel_80/3539860190.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  after = pd.read_sql_query(


## 7. Business-facing mart
The mart provides daily weather summaries by city for reporting and analysis.

In [7]:
mart_df = pd.read_sql_query(
    "SELECT * FROM analytics.fct_city_daily WHERE date = '2026-09-19' ORDER BY city_name;",
    connection
)
mart_df

/tmp/ipykernel_80/171181912.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mart_df = pd.read_sql_query(


,city_name,date,avg_temperature_max,avg_temperature_min,avg_temperature_mean,total_precipitation,max_wind_speed
0,Bengaluru,2026-09-19,30.5,20.7,25.3,11.1,9.4
1,Chennai,2026-09-19,33.9,26.1,29.6,4.8,15.3
2,Delhi,2026-09-19,32.8,25.0,28.9,0.0,5.8
3,Hyderabad,2026-09-19,32.1,25.2,28.6,1.1,8.0
4,Mumbai,2026-09-19,30.3,24.7,27.5,6.4,13.8


## Conclusion
The walkthrough demonstrates extraction and loading, dbt transformation and testing, idempotent reruns, and a business-facing daily weather mart.